In [ ]:
# Clone the BioMedKG repository containing the graph data with node features
!git clone https://github.com/hysonlab/biomedkg.git

# Navigate to the directory and list the contents to find data paths
%cd biomedkg
!ls -lh

# Note: Please check the cloned repository's README or data folder to locate
# the specific node features and graph structure files (e.g., .pt, .csv, or .pkl files)
# that you can load into your PyTorch Geometric (PyG) or DGL pipeline.

Cloning into 'biomedkg'...
remote: Enumerating objects: 796, done.
remote: Counting objects: 100% (543/543), done.
remote: Compressing objects: 100% (272/272), done.
remote: Total 796 (delta 326), reused 433 (delta 271), pack-reused 253 (from 1)
Receiving objects: 100% (796/796), 1.30 MiB | 14.74 MiB/s, done.
Resolving deltas: 100% (481/481), done.
/content/biomedkg
total 60K
drwxr-xr-x 2 root root 4.0K Oct  2 14:42 assets
drwxr-xr-x 5 root root 4.0K Oct  2 14:42 biomedkg
drwxr-xr-x 5 root root 4.0K Oct  2 14:42 configs
-rw-r--r-- 1 root root  416 Oct  2 14:42 Makefile
-rw-r--r-- 1 root root 4.4K Oct  2 14:42 ml_exp.py
-rw-r--r-- 1 root root  526 Oct  2 14:42 pyproject.toml
-rw-r--r-- 1 root root 6.7K Oct  2 14:42 README.md
drwxr-xr-x 3 root root 4.0K Oct  2 14:42 scripts
-rw-r--r-- 1 root root 1.2K Oct  2 14:42 test_dpi.py
-rw-r--r-- 1 root root 1.2K Oct  2 14:42 test_kge.py
-rw-r--r-- 1 root root 3.1K Oct  2 14:42 train_dpi.py
-rw-r--r-- 1 root root 3.5K Oct  2 14:42 train_gcl.py
-rw

In [ ]:
!huggingface-cli download tienda02/BioMedKG --repo-type=dataset --local-dir ./data

In [ ]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install -q huggingface_hub torch torch-geometric pandas numpy

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
from huggingface_hub import snapshot_download
from pathlib import Path

# ==========================================
# 2. DOWNLOAD DATASET FROM HUGGING FACE
# ==========================================
print("Downloading BioMedKG dataset...")
dataset_dir = "./data"

snapshot_download(
    repo_id="tienda02/BioMedKG",
    repo_type="dataset",
    local_dir=dataset_dir,
)
print(f"Dataset downloaded to {dataset_dir}!\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Dataset downloaded to ./BioMedKG_data!



2. embed/ (Base Language Model Embeddings)
This folder contains the raw mathematical vectors generated by passing the modalities/ text and sequences through Transformer Language Models (like BioBERT or SciBERT).

primekg_modality_lm.pickle: Contains the full set of PrimeKG nodes (diseases, proteins, etc.). This is where you saw the [N, 2, 768] shape. The "2" represents the two different data modalities encoded for each node (e.g., sequence + text description), each generating a 768-dimensional vector.

dpi_modality_lm.pickle: A smaller, filtered version containing only the embeddings for the drugs and proteins used in the specific DPI benchmark.

3. gcl_embed/ (Graph Contrastive Learning Embeddings)
This folder contains advanced, graph-aware embeddings. The authors took the base embeddings from the embed/ folder and trained them through Graph Contrastive Learning (GCL) to make nodes that are close together in the graph have similar vectors.

dgi_*.pickle, grace_*.pickle, ggd_*.pickle: These files represent different GCL training algorithms (Deep Graph Infomax, GRACE, Graph Group Discrimination) and fusion methods (attention, none, re-daf). You can swap these out in your PyTorch Geometric model to see which embedding strategy yields the best link prediction accuracy.

In [ ]:
# ==========================================
# 1. LOAD REQUIRED LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import torch
import pickle
from torch_geometric.data import HeteroData
from pathlib import Path

dataset_dir = Path("./BioMedKG_data")
graph = HeteroData()

# ==========================================
# 2. LOAD PRIME-KG EMBEDDINGS (Fixing the TypeError)
# ==========================================
# Switch to the PrimeKG pickle which contains the diseases and proteins
embed_file = dataset_dir / "embed" / "primekg_modality_lm.pickle"

print("Loading PrimeKG multimodal embeddings...")
with open(embed_file, 'rb') as f:
    embeddings = pickle.load(f)

# ==========================================
# 3. IDENTIFY DISEASES VS PROTEINS
# ==========================================
print("Mapping keys to node types...")
# Read the metadata CSVs to find the identifiers
dis_df = pd.read_csv(dataset_dir / "modalities" / "disease_feature_base.csv", low_memory=False)
prot_df = pd.read_csv(dataset_dir / "modalities" / "protein_aminoacid_sequence.csv", low_memory=False)

# Flatten all string columns into a fast-lookup set to match against pickle keys
prot_ids = set(prot_df.select_dtypes(include=['object']).values.flatten())
dis_ids = set(dis_df.select_dtypes(include=['object']).values.flatten())

# Dictionaries to store the continuous integer IDs required by PyG
node_mappings = {'gene/protein': {}, 'disease': {}}
features_list = {'gene/protein': [], 'disease': []}

for key, vector in embeddings.items():
    # Fix for TypeError: Skip any keys that are floats (like NaN)
    if not isinstance(key, str):
        continue

    # Check which node type this key belongs to
    if key in prot_ids:
        node_mappings['gene/protein'][key] = len(node_mappings['gene/protein'])
        features_list['gene/protein'].append(vector)

    elif key in dis_ids:
        node_mappings['disease'][key] = len(node_mappings['disease'])
        features_list['disease'].append(vector)

# ==========================================
# 4. BUILD HETERODATA GRAPH NODES
# ==========================================
print("Populating PyG Graph Nodes...")
for n_type in ['gene/protein', 'disease']:
    if features_list[n_type]:
        # Convert list of arrays to a single PyTorch tensor matrix
        tensor = torch.tensor(np.array(features_list[n_type]), dtype=torch.float32)
        graph[n_type].x = tensor
        graph[n_type].num_nodes = tensor.shape[0]
        print(f"  [SUCCESS] Loaded {tensor.shape[0]} nodes with {tensor.shape[1]}D features for '{n_type}'.")

print("\n=== Graph Nodes Initialized ===")
print(graph)

Loading PrimeKG multimodal embeddings...


/tmp/ipykernel_2020/648548176.py:22: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  embeddings = pickle.load(f)


Mapping keys to node types...
Populating PyG Graph Nodes...
  [SUCCESS] Loaded 20013 nodes with 2D features for 'gene/protein'.
  [SUCCESS] Loaded 22205 nodes with 2D features for 'disease'.

=== Graph Nodes Initialized ===
HeteroData(
  gene/protein={
    x=[20013, 2, 768],
    num_nodes=20013,
  },
  disease={
    x=[22205, 2, 768],
    num_nodes=22205,
  }
)


In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from IPython.display import display

dataset_dir = Path("./BioMedKG_data")

# ==========================================
# 1. LOAD IDENTIFIERS TO CLASSIFY NODES
# ==========================================
print("Loading node identifiers for classification...")
dis_df = pd.read_csv(dataset_dir / "modalities" / "disease_feature_base.csv", low_memory=False)
prot_df = pd.read_csv(dataset_dir / "modalities" / "protein_aminoacid_sequence.csv", low_memory=False)

prot_ids = set(prot_df.select_dtypes(include=['object']).values.flatten())
dis_ids = set(dis_df.select_dtypes(include=['object']).values.flatten())

# ==========================================
# 2. DEFINE THE 4 VERSIONS TO COMPARE
# ==========================================
# We use the 'attention' fusion variant for the GCL models as a baseline
files_to_compare = {
    "Base LM": dataset_dir / "embed" / "primekg_modality_lm.pickle",
    "GCL - DGI": dataset_dir / "gcl_embed" / "dgi_attention.pickle",
    "GCL - GRACE": dataset_dir / "gcl_embed" / "grace_attention.pickle",
    "GCL - GGD": dataset_dir / "gcl_embed" / "ggd_attention.pickle"
}

# ==========================================
# 3. ANALYZE AND COMPARE
# ==========================================
print("Analyzing embedding files (this may take a moment)...\n")
results = []

for name, filepath in files_to_compare.items():
    if not filepath.exists():
        print(f"[!] File not found: {filepath}")
        continue

    with open(filepath, 'rb') as f:
        data = pickle.load(f)

    disease_count = 0
    protein_count = 0
    other_count = 0
    example_shape = None

    for key, vector in data.items():
        # Skip invalid/NaN keys
        if not isinstance(key, str):
            continue

        # Capture the shape of the first valid vector we find
        if example_shape is None:
            example_shape = tuple(np.array(vector).shape)

        # Classify the node
        if key in dis_ids:
            disease_count += 1
        elif key in prot_ids:
            protein_count += 1
        else:
            other_count += 1

    results.append({
        "Embedding Version": name,
        "Feature Shape": str(example_shape),
        "Disease Nodes": disease_count,
        "Protein Nodes": protein_count,
        "Other Nodes": other_count,
        "Total Valid Nodes": disease_count + protein_count + other_count
    })

# ==========================================
# 4. DISPLAY RESULTS
# ==========================================
comparison_df = pd.DataFrame(results)
display(comparison_df)

Loading node identifiers for classification...
Analyzing embedding files (this may take a moment)...



/tmp/ipykernel_3259/4192418963.py:42: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


,Embedding Version,Feature Shape,Disease Nodes,Protein Nodes,Other Nodes,Total Valid Nodes
0,Base LM,"(2, 768)",22205,20013,15170,57388
1,GCL - DGI,"(1, 256)",17007,17755,4878,39640
2,GCL - GRACE,"(1, 256)",17007,17755,4878,39640
3,GCL - GGD,"(1, 256)",17007,17755,4878,39640


In [ ]:
path = dataset_dir / "gcl_embed" / "ggd_attention.pickle"
with open(path, 'rb') as f:
    data = pickle.load(f)


In [13]:
len(data)

39640

In [20]:
for key, vector in data.items():
    print(key)
    print((vector.shape))
    break

A1BG
(1, 256)


In [21]:
kg_csv = "/content/kg_filtered.csv"
kg = pd.read_csv(kg_csv)
kg.head()

/tmp/ipykernel_932/3127716633.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg = pd.read_csv(kg_csv)


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,protein_protein,ppi,8889,56992,gene/protein,KIF15,NCBI,0,9796,gene/protein,PHYHIP,NCBI
1,protein_protein,ppi,1,7918,gene/protein,GPANK1,NCBI,2798,9240,gene/protein,PNMA1,NCBI
2,protein_protein,ppi,5646,23548,gene/protein,TTC33,NCBI,2,8233,gene/protein,ZRSR2,NCBI
3,protein_protein,ppi,11592,11253,gene/protein,MAN1B1,NCBI,3,4899,gene/protein,NRF1,NCBI
4,protein_protein,ppi,4,5297,gene/protein,PI4KA,NCBI,2122,8601,gene/protein,RGS20,NCBI


In [39]:
# Extract x and y pairs separately
x_df = kg[["x_name", "x_type"]].rename(columns={"x_name": "name", "x_type": "type"})
y_df = kg[["y_name", "y_type"]].rename(columns={"y_name": "name", "y_type": "type"})

# Concatenate them vertically
combined_df = pd.concat([x_df, y_df], ignore_index=True)

# Drop NaNs and keep only unique name-type pairs
unique_df = combined_df.dropna().drop_duplicates().reset_index(drop=True)

# Display the unique DataFrame
unique_df

,name,type
0,KIF15,gene/protein
1,GPANK1,gene/protein
2,TTC33,gene/protein
3,MAN1B1,gene/protein
4,PI4KA,gene/protein
...,...,...
129352,NDUFB8P1,gene/protein
129353,UBL5P4,gene/protein
129354,LINC00563,gene/protein
129355,RYKP1,gene/protein


In [40]:
# Map name to its type using unique_df for quick O(1) lookup
name_to_type = dict(zip(unique_df["name"], unique_df["type"]))

type_counts = {}
missing_count = 0

for key, vector in data.items():
    if key in name_to_type:
        node_type = name_to_type[key]
        type_counts[node_type] = type_counts.get(node_type, 0) + 1
    else:
        missing_count += 1

# Display the results
print("=== Node Type Distribution in 'data' ===")
for node_type, count in type_counts.items():
    print(f"{node_type}: {count}")

print(f"Not found in unique_df: {missing_count}")

=== Node Type Distribution in 'data' ===
gene/protein: 18352
exposure: 52
drug: 4227
pathway: 1
disease: 17000
effect/phenotype: 5
anatomy: 3
Not found in unique_df: 0


In [43]:
# 1. Telepítsük a PyKeen könyvtárat, ha még nincs fent
!pip install -q pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 28.2 MB/s eta 0:00:00


In [44]:
import torch
import pandas as pd
import numpy as np
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.nn.init import PretrainedInitializer

# =========================================================================
# 2. TRIPPLETEK ELŐKÉSZÍTÉSE PYKEEN SZÁMÁRA
# =========================================================================
# Kiválasztjuk a szükséges oszlopokat (fej, reláció, farok)
triples_df = kg[['x_name', 'relation', 'y_name']].dropna().astype(str)

# Létrehozzuk a PyKeen TriplesFactory objektumot
triples_factory = TriplesFactory.from_labeled_triples(
    triples=triples_df.values,
    create_inverse_triples=False
)

# Felosztjuk a tripleteket tréning és teszt halmazra (pl. 80-20% arányban)
training_factory, testing_factory = triples_factory.split([0.8, 0.2], random_state=42)

print(f"Összes entitás száma a gyárban: {triples_factory.num_entities}")
print(f"Összes reláció száma: {triples_factory.num_relations}")

INFO:pykeen.utils:Using opt_einsum
INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [3119747, 810000]


Összes entitás száma a gyárban: 129262
Összes reláció száma: 30


In [ ]:
# =========================================================================
# 3. KÜLSŐ BEÁGYAZÁSOK (FEATURES) HOZZÁRENDELÉSE AZ ENTITÁSOKHOZ
# =========================================================================
# A PyKeen saját belső numerikus ID-kat rendel az entitás nevekhez (0-tól N-ig).
# Fontos, hogy a beágyazási mátrixunk pontosan kövesse ezt a belső sorrendet!

num_entities = training_factory.num_entities
# Lekérjük az első reprezentáció dimenzióját a meglévő pickle adatokból (pl. 256 vagy 768)
sample_vector = next(iter(data.values()))
embedding_dim = sample_vector.squeeze().shape[0]  # Eltávolítjuk a felesleges 1-es dimenziót

# Létrehozunk egy üres tenzort a külső beágyazásoknak
pretrained_embeddings = np.zeros((num_entities, embedding_dim), dtype=np.float32)

# Feltöltjük a tenzort a PyKeen entitás-név -> ID szótára alapján
entity_to_id = training_factory.entity_to_id
mapped_count = 0

for entity_name, entity_id in entity_to_id.items():
    if entity_name in data:
        # Ha az entitás szerepel a betöltött pickle fájlban, kimásoljuk a vektort
        vec = data[entity_name].squeeze()
        pretrained_embeddings[entity_id] = vec
        mapped_count += 1
    else:
        # Ha nincs hozzá beágyazásunk, inicializálhatjuk véletlenszerűen is
        pretrained_embeddings[entity_id] = np.random.normal(0, 0.1, size=(embedding_dim,))

# Átalakítás PyTorch tenzorrá
pretrained_tensor = torch.tensor(pretrained_embeddings, dtype=torch.float32)

print(f"Sikeresen hozzárendelt entitások száma: {mapped_count} / {num_entities}")
print(f"Beágyazási mátrix alakja: {pretrained_tensor.shape}")
# =========================================================================
# 4. MODELL TANÍTÁSA A KÜLSŐ FEATURES FELHASZNÁLÁSÁVAL
# =========================================================================
# A PyKeen pipeline segítségével indítjuk a tanítást.
# A 'pretrained_initializer'-en keresztül adjuk át az előre elkészített tenzort.

result = pipeline(
    training=training_factory,
    testing=testing_factory,
    model='TransE',  # Használhatsz más modellt is, pl. RotatE, DistMult, ComplEx
    model_kwargs=dict(
        embedding_dim=embedding_dim,
        entity_initializer=PretrainedInitializer(tensor=pretrained_tensor),
    ),
    training_kwargs=dict(
        num_epochs=5,  # Állítsd be a kívánt epoch számot
        batch_size=256,
        use_tqdm=True
    ),
    evaluation_kwargs=dict(
        use_tqdm=True
    ),
    device='cuda' if torch.cuda.is_available() else 'cpu' # GPU használata, ha elérhető
)

# Eredmények mentése és megjelenítése
print("Tanítás befejeződött!")
result.plot_losses()

Sikeresen hozzárendelt entitások száma: 39640 / 129262
Beágyazási mátrix alakja: torch.Size([129262, 256])


In [ ]:
import torch
from pykeen.models import ERModel
from pykeen.nn.representation import CombinedRepresentation, Embedding
from pykeen.nn.init import PretrainedInitializer

feature_dim = 64
learned_dim = 64

# Feature-based representation (frozen or trainable)
feature_rep = Embedding(
    max_id=dataset.num_entities,
    shape=feature_dim,
    initializer=PretrainedInitializer(tensor=pretrained_features),
    trainable=False,
)

# Learnable structural representation
learned_rep = Embedding(
    max_id=dataset.num_entities,
    shape=learned_dim,
)

# Combine both representations by concatenation
combined_entity_rep = CombinedRepresentation(
    max_id=dataset.num_entities,
    base=[feature_rep, learned_rep],
    combination="concat",  # Options: 'concat', 'add', or custom module
)

model = ERModel(
    triples_factory=dataset.training,
    interaction="TransE",
    entity_representations=combined_entity_rep,
)